# Calibration Report Generator — HEC-RAS Example 24

This notebook demonstrates the `calibration_report` module using HEC-RAS **Example 24 — Manning's n Calibration** (Mississippi/Ohio River confluence, Sep 2004–Mar 2005).

The example ships with HEC-RAS and includes:
- **11 observed gauge stations** from LMRFC (Lower Mississippi River Forecast Center) in `LMRFC.dss`
- **5 computed plans** in `Manning'snCalibra.dss`: No Calibration, Auto Calibrate, Sequential, Squared Error, and Final Model
- Stage time series at 6-hour intervals across Mississippi Upper/Lower and Ohio River reaches

The report generator produces a **self-contained HTML file** with interactive Bokeh plots and DSS-Commander-style color-coded calibration statistics (RMSE%, Pearson r, PBIAS, NSE, KGE).

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd

# ras-commander for DSS reading
sys.path.insert(0, str(Path(r"G:\GH\ras-commander")))
from ras_commander.dss import RasDss

# calibration_report module (from this repo's pipeline/)
sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "pipeline") if "__file__" in dir() else str(Path.cwd().parent / "pipeline"))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "pipeline"))
pipeline_path = Path(r"G:\GH\symphony-workspaces\ras-agent\CLB-423\pipeline")
sys.path.insert(0, str(pipeline_path))

import calibration_report
from calibration_report import GaugeRecord, generate_calibration_report, calculate_stats

print(f"RasDss loaded from: {RasDss.__module__}")
print(f"calibration_report loaded from: {calibration_report.__file__}")

## Configuration

Point to the Example 24 project directory and define the simulation window. The gauge-to-station mapping comes from the unsteady flow file (`Manning'snCalibra.u01`).

In [ ]:
EX24_DIR = Path(r"C:\Users\bill\Documents\HEC Data\HEC-RAS\Example Data\Applications Guide\Chapter 24 - Mannings-n-Calibration")

OBSERVED_DSS = str(EX24_DIR / "LMRFC.dss")
MODELED_DSS = str(EX24_DIR / "Manning'snCalibra.dss")

SIM_START = pd.Timestamp("2004-09-25 12:00")
SIM_END = pd.Timestamp("2005-03-05 12:00")

DSS_MISSING = -3.4028e+38

PLANS_TO_COMPARE = ["FINAL MODEL", "NOCALIBRATION"]

# Gauge-to-station mapping extracted from Manning'snCalibra.u01
# Each entry: (display_name, gauge_id, observed_dss_pathname, river_reach_part_a, station_part_b)
GAUGE_MAP = [
    ("Chester IL (CHSI2)",      "CHSI2", "/CHSI2/CHSI2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI UPPER MISS", "110.4"),
    ("Thebes IL (THBI2)",       "THBI2", "/THBI2/THBI2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI UPPER MISS", "43.7"),
    ("Price Landing (PCLM7)",   "PCLM7", "/PCLM7/PCLM7/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI UPPER MISS", "28.57"),
    ("Cairo IL (CIRI2)",        "CIRI2", "/CIRI2/CIRI2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI LOWER MISS.", "953.5"),
    ("Hickman KY (HKMK2)",      "HKMK2", "/HKMK2/HKMK2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI LOWER MISS.", "922"),
    ("Tiptonville TN (TPTT1)",  "TPTT1", "/TPTT1/TPTT1/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI LOWER MISS.", "873"),
    ("Paducah KY (PAHK2)",      "PAHK2", "/PAHK2/PAHK2/STAGE/01SEP2004/6HOUR/OBS/",  "OHIO RIVER OHS",         "-935.66"),
    ("Metropolis IL (MTRI2)",   "MTRI2", "/MTRI2/MTRI2/STAGE/01SEP2004/6HOUR/OBS/",  "OHIO RIVER OHS",         "-944.1"),
    ("Smithland KY (SMLI2)",    "SMLI2", "/SMLI2/SMLI2/FLOW/01SEP2004/6HOUR/OBS/",   "OHIO RIVER OHS",         "-961"),
]

print(f"Project: {EX24_DIR.name}")
print(f"Observed DSS: {Path(OBSERVED_DSS).name} ({Path(OBSERVED_DSS).stat().st_size / 1e6:.1f} MB)")
print(f"Modeled DSS:  {Path(MODELED_DSS).name} ({Path(MODELED_DSS).stat().st_size / 1e6:.1f} MB)")
print(f"Simulation:   {SIM_START} to {SIM_END}")
print(f"Gauges:       {len(GAUGE_MAP)}")
print(f"Plans:        {PLANS_TO_COMPARE}")

## Read Observed Gauge Data

Read each gauge's observed stage time series from `LMRFC.dss` and clip to the simulation window. DSS missing-value sentinels (`-3.4028e+38`) are filtered out.

In [ ]:
def read_dss_series(dss_file, pathname, start, end, missing_sentinel=-3.4028e+38):
    """Read a DSS time series, filter missing values, and clip to date range."""
    df = RasDss.read_timeseries(dss_file, pathname)
    series = df["value"]
    series = series[series > missing_sentinel * 0.9]  # remove sentinel values
    series = series[(series.index >= start) & (series.index <= end)]
    return series

observed_data = {}
for name, gauge_id, obs_path, river_reach, station in GAUGE_MAP:
    try:
        obs = read_dss_series(OBSERVED_DSS, obs_path, SIM_START, SIM_END)
        observed_data[name] = {
            "gauge_id": gauge_id,
            "river_reach": river_reach,
            "station": station,
            "observed": obs,
        }
        print(f"  {name:30s}  {len(obs):5d} pts  range: {obs.min():.1f} - {obs.max():.1f}")
    except Exception as e:
        print(f"  {name:30s}  FAILED: {e}")

print(f"\nLoaded {len(observed_data)} / {len(GAUGE_MAP)} observed gauge series")

## Read Modeled Data

For each gauge location, read the modeled stage from the main DSS for each plan. The DSS pathname pattern is `/{river_reach}/{station}/STAGE/{date}/6HOUR/{plan_name}/`.

In [ ]:
for name, info in observed_data.items():
    river_reach = info["river_reach"]
    station = info["station"]
    info["modeled"] = {}
    
    for plan in PLANS_TO_COMPARE:
        modeled_path = f"/{river_reach}/{station}/STAGE/01SEP2004/6HOUR/{plan}/"
        try:
            mod = read_dss_series(MODELED_DSS, modeled_path, SIM_START, SIM_END)
            info["modeled"][plan] = mod
            print(f"  {name:30s}  {plan:20s}  {len(mod):5d} pts")
        except Exception as e:
            print(f"  {name:30s}  {plan:20s}  FAILED: {e}")

loaded = sum(1 for info in observed_data.values() if info["modeled"])
print(f"\nLoaded modeled data for {loaded} / {len(observed_data)} gauges")

## Preview: Per-Gauge Calibration Statistics

Before generating the full HTML report, compute and display the DSS-Commander-style metrics for each gauge and plan. Thresholds: RMSE% <= 15%, r >= 0.9, PBIAS +/-10%, NSE >= 0.6.

In [ ]:
rows = []
for name, info in observed_data.items():
    obs = info["observed"]
    for plan_label, mod in info.get("modeled", {}).items():
        try:
            stats = calculate_stats(obs, mod, variable="stage")
            rows.append({
                "Gauge": name,
                "Plan": plan_label,
                "N": len(pd.concat([obs.rename("o"), mod.rename("m")], axis=1, join="inner").dropna()),
                "RMSE%": round(stats["rmse_pct"], 1),
                "r": round(stats["pearson_r"], 3),
                "PBIAS%": round(stats["pbias"], 1),
                "NSE": round(stats["nse"], 3),
                "KGE": round(stats["kge"], 3),
            })
        except Exception as e:
            rows.append({"Gauge": name, "Plan": plan_label, "Error": str(e)})

stats_df = pd.DataFrame(rows)
stats_df

## Generate Calibration Report

Build `GaugeRecord` objects and call `generate_calibration_report()` to produce a self-contained HTML file with interactive Bokeh plots and color-coded statistics tables.

In [ ]:
gauge_records = []
for name, info in observed_data.items():
    if not info.get("modeled"):
        continue
    gauge_records.append({
        "name": name,
        "observed": info["observed"],
        "modeled": info["modeled"],
        "variable": "stage",
        "units": "ft",
    })

output_path = Path("calibration_report_example24.html")

report_path = generate_calibration_report(
    plan_hdfs=[],
    observed_data=gauge_records,
    output_path=output_path,
)

file_size_kb = report_path.stat().st_size / 1024
print(f"Report written: {report_path}")
print(f"File size: {file_size_kb:.0f} KB")
print(f"Gauges: {len(gauge_records)}, Plans: {len(PLANS_TO_COMPARE)}")

## Open Report

Open the generated HTML file in the default browser.

In [ ]:
import webbrowser
webbrowser.open(str(report_path.resolve()))
print(f"Opened: {report_path.resolve()}")